In [2]:
import os
import shutil


source = "../dataset/raw/Non-Recyclable"

destination = "../dataset/non_recyclable_dataset"


os.makedirs(
    destination,
    exist_ok=True
)


for category in os.listdir(source):

    category_path = os.path.join(
        source,
        category
    )


    if os.path.isdir(category_path):

        destination_path = os.path.join(
            destination,
            category
        )


        os.makedirs(
            destination_path,
            exist_ok=True
        )


        for image in os.listdir(category_path):

            shutil.copy(

                os.path.join(
                    category_path,
                    image
                ),

                os.path.join(
                    destination_path,
                    image
                )
            )


print("Non-Recyclable dataset created")

Non-Recyclable dataset created


In [3]:
non_recyclable_path = "../dataset/non_recyclable_dataset"


print(
    os.listdir(non_recyclable_path)
)

['ceramic_product', 'diapers', 'platics_bags_wrappers', 'sanitary_napkin', 'stroform_product']


In [4]:
for folder in os.listdir(non_recyclable_path):

    path = os.path.join(
        non_recyclable_path,
        folder
    )

    print(
        folder,
        len(os.listdir(path))
    )

ceramic_product 139
diapers 145
platics_bags_wrappers 135
sanitary_napkin 110
stroform_product 118


In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


train_datagen = ImageDataGenerator(

    preprocessing_function=preprocess_input,

    validation_split=0.2,

    rotation_range=20,

    zoom_range=0.2,

    width_shift_range=0.15,

    height_shift_range=0.15,

    horizontal_flip=True,

    brightness_range=[0.8,1.2]

)


val_datagen = ImageDataGenerator(

    preprocessing_function=preprocess_input,

    validation_split=0.2

)

In [6]:
IMG_SIZE=(224,224)

BATCH_SIZE=32


train_data = train_datagen.flow_from_directory(

    non_recyclable_path,

    target_size=IMG_SIZE,

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    subset="training"

)


val_data = val_datagen.flow_from_directory(

    non_recyclable_path,

    target_size=IMG_SIZE,

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    subset="validation",

    shuffle=False

)

Found 516 images belonging to 5 classes.
Found 126 images belonging to 5 classes.


In [7]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,GlobalAveragePooling2D


base_model = MobileNetV2(

    weights="imagenet",

    include_top=False,

    input_shape=(224,224,3)

)


base_model.trainable=False

In [8]:
non_recyclable_model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(
        128,
        activation="relu"
    ),

    Dropout(0.5),

    Dense(
        5,
        activation="softmax"
    )

])


non_recyclable_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │             645 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,422,597 (9.24 MB)

 Trainable params: 164,613 (643.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
from tensorflow.keras.optimizers import Adam


non_recyclable_model.compile(

    optimizer=Adam(
        learning_rate=0.0001
    ),

    loss="categorical_crossentropy",

    metrics=[
        "accuracy"
    ]

)

In [10]:
history = non_recyclable_model.fit(

    train_data,

    validation_data=val_data,

    epochs=20

)

Epoch 1/20
 6/17 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.2188 - loss: 2.4868

C:\Disk D\Year 2 Semester 1\Year2Semester1\Year2Semester1\IT2011 AI & ML\AI ML Project\Waste-Classification-AI\venv\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


11/17 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.2301 - loss: 2.3127

C:\Disk D\Year 2 Semester 1\Year2Semester1\Year2Semester1\IT2011 AI & ML\AI ML Project\Waste-Classification-AI\venv\Lib\site-packages\PIL\Image.py:1136: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


17/17 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step - accuracy: 0.2306 - loss: 2.1916 - val_accuracy: 0.2778 - val_loss: 1.6569
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 0.2791 - loss: 1.8774 - val_accuracy: 0.3413 - val_loss: 1.5109
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 0.3411 - loss: 1.6305 - val_accuracy: 0.4127 - val_loss: 1.4106
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 0.3353 - loss: 1.6015 - val_accuracy: 0.3968 - val_loss: 1.3396
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 0.4070 - loss: 1.4254 - val_accuracy: 0.4683 - val_loss: 1.2921
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 0.4535 - loss: 1.3575 - val_accuracy: 0.4921 - val_loss: 1.2298
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 0.5000 - loss: 1.2548 - val_accuracy: 0.5317 - val_loss: 1.1870
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 0.5155 - loss: 1.2163 - val_accuracy: 0.5635 - val_loss: 1.1441
Epo

In [11]:
base_model = non_recyclable_model.layers[0]

base_model.trainable = True


for layer in base_model.layers[:-50]:
    layer.trainable = False


print("Fine tuning enabled")

Fine tuning enabled


In [12]:
from tensorflow.keras.optimizers import Adam


non_recyclable_model.compile(

    optimizer=Adam(
        learning_rate=0.00001
    ),

    loss="categorical_crossentropy",

    metrics=[
        "accuracy"
    ]

)

In [13]:
fine_history = non_recyclable_model.fit(

    train_data,

    validation_data=val_data,

    epochs=10

)

Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.5543 - loss: 1.1613 - val_accuracy: 0.7222 - val_loss: 0.8983
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.5930 - loss: 1.0873 - val_accuracy: 0.7143 - val_loss: 0.8925
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.5911 - loss: 1.0794 - val_accuracy: 0.7222 - val_loss: 0.8858
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step - accuracy: 0.6202 - loss: 1.0003 - val_accuracy: 0.7143 - val_loss: 0.8816
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step - accuracy: 0.6105 - loss: 1.0533 - val_accuracy: 0.7143 - val_loss: 0.8767
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step - accuracy: 0.6512 - loss: 0.9284 - val_accuracy: 0.7143 - val_loss: 0.8744
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.6783 - loss: 0.9129 - val_accuracy: 0.7143 - val_loss: 0.8725
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step - accuracy: 0.6899 - loss: 0.8783 - val_accuracy: 0.7143 - val_loss:

In [14]:
non_recyclable_model.save(
    "../models/non_recyclable_model.keras"
)

print("Non-Recyclable model saved")

Non-Recyclable model saved
